In [ ]:
"""
因子 #14：开盘30分钟 VWAP / 全天 VWAP（早盘信号质量因子）
============================================================
赛道：传统量化
经济逻辑：
    开盘前30分钟（09:30-10:00）是隔夜信息和开盘集合竞价
    消化最密集的时段。这段时间的价格和成交量包含了
    最"新鲜"的信息。

    开盘30min VWAP / 全天 VWAP > 1 → 开盘均价高于全天
    → 开盘强势但随后走弱 → 日内反转信号 → 次日反弹（看涨）

    开盘30min VWAP / 全天 VWAP < 1 → 开盘均价低于全天
    → 开盘弱势但随后走强 → 日内趋势延续（因子值小，看跌）

    这是 VWAP 系列的第 4 个因子，聚焦开盘窗口的信息含量。

因子方向：因子值越大（早盘强→随后走弱→反转）
         → 预期未来收益越高（正向）
============================================================
"""

from __future__ import annotations


def main(datasources, start_date, end_date):
    import dai
    import pandas as pd
    import numpy as np

    table_name = datasources["bar1m"]

    # 保留完整时间戳判断开盘30分钟
    sql = f"""
        SELECT
            date AS date,
            instrument,
            close,
            volume
        FROM {table_name}
    """

    raw = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    raw["datetime"] = pd.to_datetime(raw["date"])
    raw["minutes"] = raw["datetime"].dt.hour * 60 + raw["datetime"].dt.minute
    raw["date"] = raw["datetime"].dt.normalize()

    # 开盘30分钟 = 09:30-10:00 → 分钟数 570-600
    is_open30 = (raw["minutes"] >= 570) & (raw["minutes"] <= 600)

    raw["amt"] = raw["close"] * raw["volume"]
    raw["open30_amt"] = np.where(is_open30, raw["amt"], 0.0)
    raw["open30_vol"] = np.where(is_open30, raw["volume"], 0)

    daily = raw.groupby(["date", "instrument"]).agg(
        open30_amt=("open30_amt", "sum"),
        open30_vol=("open30_vol", "sum"),
        total_amt=("amt", "sum"),
        total_vol=("volume", "sum"),
    ).reset_index()

    # 开盘30min VWAP
    daily["vwap_open30"] = np.where(
        daily["open30_vol"] > 0,
        daily["open30_amt"] / daily["open30_vol"],
        np.nan,
    )
    # 全天 VWAP
    daily["vwap_full"] = np.where(
        daily["total_vol"] > 0,
        daily["total_amt"] / daily["total_vol"],
        np.nan,
    )

    # 开盘30min / 全天 VWAP
    daily["raw_factor"] = np.where(
        daily["vwap_full"].notna() & (daily["vwap_full"] > 0),
        daily["vwap_open30"] / daily["vwap_full"],
        np.nan,
    )

    daily["raw_factor"] = daily["raw_factor"].replace([np.inf, -np.inf], np.nan)
    daily = daily.dropna(subset=["raw_factor"])

    def normalize(group):
        raw = group["raw_factor"]
        med = raw.median()
        mad = (raw - med).abs().median()
        if mad == 0 or pd.isna(mad):
            mad = 1e-8
        clipped = raw.clip(lower=med - 5 * mad, upper=med + 5 * mad)
        std = clipped.std()
        if std == 0 or pd.isna(std):
            std = 1e-8
        group["factor"] = (clipped - clipped.mean()) / std
        return group

    daily = daily.groupby("date", group_keys=False).apply(normalize)
    daily["factor"] = pd.to_numeric(daily["factor"], errors="coerce")

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"]).dt.normalize()

    return (
        pd.merge(daily, stk_pool, how="inner", on=["date", "instrument"])
        .dropna(subset=["factor"])
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
        .loc[:, ["date", "instrument", "factor"]]
    )
